In [0]:
dbutils.library.restartPython()

In [0]:
import os
import sys

# os.getcwd() returns the folder containing THIS notebook (DBR 11.0+)
notebook_dir = os.getcwd()

# Walk up to the repo root, then into scripts/.
# Adjust the number of ".." depending on how deep your notebook is nested
# e.g. if notebook is at enterprise-mlops/notebooks/my_notebook, one ".." gets you to the repo root

repo_root = os.path.abspath(os.path.join(notebook_dir, ".."))
scripts_dir = os.path.join(repo_root, "scripts")

sys.path.insert(0, scripts_dir)

from generate_mock_data import generate_dataset

df = generate_dataset(n_assets=400, readings_per_asset=40, seed=42)


In [0]:
spark_df = spark.createDataFrame(df)
spark_df.write.mode("overwrite").option("header", True).csv(
    "abfss://raw@enterprisemlopssa.dfs.core.windows.net/asset_telemetry.csv"
)

storage_account = "enterprisemlopssa"
tenant_id = dbutils.secrets.get(scope="storage-auth", key="tenant-id")

azure_storage_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": dbutils.secrets.get(scope="storage-auth", key="client-id"),
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": dbutils.secrets.get(scope="storage-auth", key="client-secret"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}




spark_df.write.options(**azure_storage_options).mode("overwrite").option("header", True).csv(
    "abfss://raw@enterprisemlopssa.dfs.core.windows.net/asset_telemetry.csv"
)